# HNDSR - Complete Pipeline (Kaggle Optimized)
This notebook contains the complete, updated pipeline for the Hybrid Neural Operator-Diffusion Model. It includes automatic dataset discovery and the fixed spatial cross-attention to resolve the random noise issue.

In [ ]:
import os
import glob
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from PIL import Image
from tqdm.auto import tqdm
import numpy as np
import matplotlib.pyplot as plt

# ==========================================
# 1. Hyperparameters & Configuration
# ==========================================
CONFIG = {
    "device": "cuda" if torch.cuda.is_available() else "cpu",
    "batch_size": 8,
    "lr": 1e-4,
    "epochs_stage3": 50,
    "latent_dim": 128,
    "num_timesteps": 1000,
    "scale_factor": 4,
    "image_size": 256,
    "save_path": "hndsr_model.pth"
}

In [ ]:
# ==========================================
# 2. Dataset & Auto-Path Finder
# ==========================================
class SatelliteDataset(Dataset):
    def __init__(self, root_dir, transform=None, scale=4):
        self.root_dir = root_dir
        self.transform = transform
        self.scale = scale
        
        # Recursively find all images to avoid the num_samples=0 error
        search_path = os.path.join(root_dir, '**', '*.*')
        self.image_files = [
            f for f in glob.glob(search_path, recursive=True) 
            if f.lower().endswith(('.png', '.jpg', '.jpeg'))
        ]
        print(f"Loaded {len(self.image_files)} images from dataset.")

    def __len__(self):
        return len(self.image_files)

    def __getitem__(self, idx):
        img_path = self.image_files[idx]
        hr_image = Image.open(img_path).convert('RGB')
        
        if self.transform:
            hr_image = self.transform(hr_image)
            
        # Generate LR image via bicubic downsampling
        c, h, w = hr_image.shape
        lr_image = F.interpolate(hr_image.unsqueeze(0), 
                                 size=(h // self.scale, w // self.scale), 
                                 mode='bicubic', align_corners=False).squeeze(0)
        
        return {"lr": lr_image, "hr": hr_image}

def find_kaggle_dataset():
    """Automatically locates the dataset folder in Kaggle."""
    for root, dirs, files in os.walk('/kaggle/input'):
        for file in files:
            if file.lower().endswith(('.png', '.jpg', '.jpeg')):
                # Return the root directory containing the first found image
                # This goes up one level if images are inside train/ or images/ folders
                return root.rsplit('/', 1)[0] if 'train' in root or 'images' in root else root
    return "/kaggle/input"

CONFIG["data_path"] = find_kaggle_dataset()
print(f"Auto-configured dataset path to: {CONFIG['data_path']}")

In [ ]:
# ==========================================
# 3. Model Architecture (With Spatial Fix)
# ==========================================
class CrossAttention(nn.Module):
    """Fixed Cross-attention for spatial conditioning"""
    def __init__(self, query_dim, context_dim, heads=8, dim_head=64):
        super().__init__()
        inner_dim = dim_head * heads
        self.scale = dim_head ** -0.5
        self.heads = heads
        self.to_q = nn.Linear(query_dim, inner_dim, bias=False)
        self.to_k = nn.Linear(context_dim, inner_dim, bias=False)
        self.to_v = nn.Linear(context_dim, inner_dim, bias=False)
        self.to_out = nn.Linear(inner_dim, query_dim)

    def forward(self, x, context):
        h = self.heads
        q = self.to_q(x)
        k = self.to_k(context)
        v = self.to_v(context)
        q, k, v = map(lambda t: t.view(t.shape[0], t.shape[1], h, -1).transpose(1, 2), (q, k, v))
        
        sim = torch.matmul(q, k.transpose(-1, -2)) * self.scale
        attn = sim.softmax(dim=-1)
        
        out = torch.matmul(attn, v)
        out = out.transpose(1, 2).reshape(x.shape[0], x.shape[1], -1)
        return self.to_out(out)

class DiffusionUNet(nn.Module):
    def __init__(self, in_channels=3, context_dim=128):
        super().__init__()
        self.init_conv = nn.Conv2d(in_channels, 128, 3, padding=1)
        self.time_mlp = nn.Sequential(nn.Linear(1, 128), nn.SiLU(), nn.Linear(128, 128))
        self.attn = CrossAttention(query_dim=128, context_dim=context_dim)
        self.final_conv = nn.Conv2d(128, in_channels, 3, padding=1)

    def forward(self, x, t, context):
        t = t.float().view(-1, 1)
        t_emb = self.time_mlp(t).view(-1, 128, 1, 1)
        h = self.init_conv(x) + t_emb
        
        b, c, h_feat, w_feat = h.shape
        h_flat = h.view(b, c, -1).transpose(1, 2) 
        ctx_flat = context.view(b, context.shape[1], -1).transpose(1, 2)
        
        h_attn = self.attn(h_flat, ctx_flat)
        h = h_attn.transpose(1, 2).view(b, c, h_feat, w_feat)
        return self.final_conv(h)

class NeuralOperator(nn.Module):
    def __init__(self, in_channels=3, out_channels=128):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_channels, 64, 3, padding=1),
            nn.ReLU(),
            nn.Conv2d(64, out_channels, 3, padding=1)
        )
    def forward(self, x):
        return self.conv(x)

class HNDSR(nn.Module):
    def __init__(self):
        super().__init__()
        self.neural_operator = NeuralOperator()
        self.diffusion_unet = DiffusionUNet(in_channels=3, context_dim=128)
        
    def forward(self, lr, t, noise_latent):
        prior = self.neural_operator(lr)
        prior = F.interpolate(prior, size=noise_latent.shape[2:], mode='bilinear')
        return self.diffusion_unet(noise_latent, t, prior)
        
    @torch.no_grad()
    def super_resolve(self, lr_img, num_inference_steps=50):
        self.eval()
        device = lr_img.device
        b = lr_img.shape[0]
        
        prior = self.neural_operator(lr_img)
        h, w = lr_img.shape[2] * 4, lr_img.shape[3] * 4
        prior = F.interpolate(prior, size=(h, w), mode='bilinear')
        
        z_t = torch.randn((b, 3, h, w), device=device)
        step_size = CONFIG["num_timesteps"] // num_inference_steps
        timesteps = torch.arange(CONFIG["num_timesteps"] - 1, 0, -step_size, device=device)
        
        for t_val in tqdm(timesteps, desc="Denoising"):
            t_batch = torch.full((b,), t_val, device=device, dtype=torch.long)
            noise_pred = self.diffusion_unet(z_t, t_batch, prior)
            alpha_t = 1 - (t_val.float() / CONFIG["num_timesteps"])
            z_t = (z_t - (1 - alpha_t) / torch.sqrt(1 - alpha_t) * noise_pred) / torch.sqrt(alpha_t)
            
        return z_t

In [ ]:
# ==========================================
# 4. Training and Inference Pipeline
# ==========================================
def train():
    transform = transforms.Compose([
        transforms.Resize((CONFIG["image_size"], CONFIG["image_size"])),
        transforms.ToTensor(),
        transforms.Normalize((0.5,), (0.5,))
    ])
    
    dataset = SatelliteDataset(CONFIG["data_path"], transform=transform)
    if len(dataset) == 0:
        print("\n[!] ERROR: No images found. Please check your Kaggle dataset.")
        return
        
    dataloader = DataLoader(dataset, batch_size=CONFIG["batch_size"], shuffle=True)
    model = HNDSR().to(CONFIG["device"])
    optimizer = torch.optim.Adam(model.parameters(), lr=CONFIG["lr"])
    
    print("\n🚀 Starting Training...")
    for epoch in range(CONFIG["epochs_stage3"]):
        model.train()
        epoch_loss = 0
        for batch in tqdm(dataloader, desc=f"Epoch {epoch+1}/{CONFIG['epochs_stage3']}"):
            lr = batch["lr"].to(CONFIG["device"])
            hr = batch["hr"].to(CONFIG["device"])
            
            t = torch.randint(0, CONFIG["num_timesteps"], (lr.shape[0],), device=CONFIG["device"])
            noise = torch.randn_like(hr)
            alpha_t = 1 - (t.float() / CONFIG["num_timesteps"]).view(-1, 1, 1, 1)
            noisy_hr = torch.sqrt(alpha_t) * hr + torch.sqrt(1 - alpha_t) * noise
            
            noise_pred = model(lr, t, noisy_hr)
            loss = F.mse_loss(noise_pred, noise)
            
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()
            
        print(f"✓ Epoch {epoch+1} Complete | Avg Loss: {epoch_loss/len(dataloader):.5f}")
        
        # Save checkpoint every 10 epochs
        if (epoch + 1) % 10 == 0:
            torch.save(model.state_dict(), CONFIG["save_path"])

    torch.save(model.state_dict(), CONFIG["save_path"])
    print(f"🎉 Training Finished! Model saved to {CONFIG['save_path']}")
    return model

@torch.no_grad()
def test_model(model_path, image_path):
    print("\n🔍 Running Inference Test...")
    model = HNDSR().to(CONFIG["device"])
    if os.path.exists(model_path):
        model.load_state_dict(torch.load(model_path, map_location=CONFIG["device"]))
        print("Loaded trained weights.")
    else:
        print("Warning: No weights found. Running with initialized model.")
        
    img = Image.open(image_path).convert('RGB')
    transform = transforms.Compose([
        transforms.Resize((CONFIG["image_size"] // 4, CONFIG["image_size"] // 4)),
        transforms.ToTensor(),
        transforms.Normalize((0.5,), (0.5,))
    ])
    lr_tensor = transform(img).unsqueeze(0).to(CONFIG["device"])
    
    sr_tensor = model.super_resolve(lr_tensor, num_inference_steps=50)
    
    sr_img = (sr_tensor.squeeze(0).cpu() * 0.5 + 0.5).clamp(0, 1).permute(1, 2, 0).numpy()
    lr_img = (lr_tensor.squeeze(0).cpu() * 0.5 + 0.5).clamp(0, 1).permute(1, 2, 0).numpy()
    
    plt.figure(figsize=(12, 6))
    plt.subplot(1, 2, 1); plt.imshow(lr_img); plt.title("Low Res Input"); plt.axis('off')
    plt.subplot(1, 2, 2); plt.imshow(sr_img); plt.title("HNDSR Super Resolved"); plt.axis('off')
    plt.show()

# Start the process
if __name__ == "__main__":
    # 1. Train the model
    train()
    
    # 2. Test the model (finds a sample image from the dataset)
    search_path = os.path.join(CONFIG["data_path"], '**', '*.*')
    sample_images = [f for f in glob.glob(search_path, recursive=True) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
    
    if sample_images:
        test_model(CONFIG["save_path"], sample_images[0])